# Haversine vs Euclidean Spatial Distance Metric Comparative Study

In [ ]:
from data_preparation import prepare_dataset, get_loss_weights
from graph_construction import build_graph
from train_and_evaluate import train_and_evaluate
from models import GoldenTransformer
from resources import (
    DATA_PATH,
    DEVICE,
    OUTPUT_TABLES_PATH,
    OUTPUT_FIGURES_PATH,
    MODELS_PATH
)
import pandas as pd
import torch
import os
import numpy as np
import random
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, matthews_corrcoef, balanced_accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

print(f"Using device: {DEVICE}")

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Approved seeds list (with 2222 replacing 333 per user request)
seeds = [111, 222, 444, 555, 666, 777, 888, 999, 1111, 2222]
num_runs = len(seeds)
print(f"Configured {num_runs} seeds for comparative runs: {seeds}")

In [ ]:
# Load dataset with raw spatial coordinates and group IDs
x, y, pos_combined, pos_spatial, pos_temporal, group_ids, pos_spatial_raw = prepare_dataset(
    DATA_PATH, 
    return_group_ids=True, 
    return_raw_spatial=True
)
class_weights = get_loss_weights(y=y)

In [ ]:
# Construct node features: Log-Area + standardized coordinates (Lat, Lon) + standardized Days
# No seasonal features are included, matching user instructions.
log_area_t = x.to(DEVICE) if x.dim() == 2 else x.unsqueeze(1).to(DEVICE)
x_full = torch.cat([
    log_area_t, 
    pos_spatial.to(DEVICE), 
    pos_temporal.to(DEVICE)
], dim=1)
print(f"Node features shape: {x_full.shape}")

In [ ]:
def calculate_binary_metrics(model, data_obj, mask):
    """ Helper to compute all 5 metrics for a GNN model """
    model.eval()
    device = next(model.parameters()).device
    
    with torch.no_grad():
        data_obj = data_obj.to(device)
        logits = model(data_obj.x, data_obj.edge_index, data_obj.edge_attr)
        y_true = data_obj.y[mask].cpu().numpy()
        probs = torch.softmax(logits[mask], dim=1)[:, 1].cpu().numpy()
        preds = logits[mask].argmax(dim=1).cpu().numpy()

    return {
        'F1': f1_score(y_true, preds, average='macro'),
        'Acc': accuracy_score(y_true, preds),
        'Balanced_Acc': balanced_accuracy_score(y_true, preds),
        'AUC': roc_auc_score(y_true, probs),
        'MCC': matthews_corrcoef(y_true, preds)
    }

In [ ]:
metrics_list = ['F1', 'Acc', 'Balanced_Acc', 'AUC', 'MCC']
results = {metric: {m: [] for m in metrics_list} for metric in ['euclidean', 'haversine']}

K_NEIGHBORS = 27
HEADS = 8
HIDDEN_DIM = 16

for metric in ['euclidean', 'haversine']:
    print("\n" + "="*60)
    print(f"EVALUATING METRIC: {metric}")
    print("="*60)
    
    # Build the graph topology using the specified distance metric
    data = build_graph(
        graph_type='multirelational',
        neighbors=K_NEIGHBORS,
        pos_spatial=pos_spatial,
        pos_temporal=pos_temporal,
        pos_combined=pos_combined,
        x=x_full,
        y=y,
        causal=True,
        group_ids=group_ids,
        distance_metric=metric,
        pos_spatial_raw=pos_spatial_raw
    ).to(DEVICE)
    
    in_dim = data.x.size(1)
    e_dim = data.edge_attr.size(1)
    
    for i, seed in enumerate(seeds):
        set_seed(seed)
        model = GoldenTransformer(
            input_dim=in_dim,
            hidden_dim=HIDDEN_DIM,
            edge_dim=e_dim,
            num_heads=HEADS
        ).to(DEVICE)
        
        _, _, trained_model = train_and_evaluate(
            model=model,
            data=data,
            device=DEVICE,
            class_weights=class_weights,
            max_epochs=500,
            patience=30
        )
        
        run_metrics = calculate_binary_metrics(trained_model, data, data.test_mask)
        for m in metrics_list:
            results[metric][m].append(run_metrics[m])
            
        print(f"Run {i+1}/{num_runs} | Seed {seed} | Test F1: {run_metrics['F1']:.4f} | Acc: {run_metrics['Acc']:.4f}")

In [ ]:
summary = {}
for m in metrics_list:
    e_mean, e_std = np.mean(results['euclidean'][m]), np.std(results['euclidean'][m])
    h_mean, h_std = np.mean(results['haversine'][m]), np.std(results['haversine'][m])
    summary[m] = {
        'Euclidean_Mean': e_mean,
        'Euclidean_SD': e_std,
        'Haversine_Mean': h_mean,
        'Haversine_SD': h_std
    }

summary_df = pd.DataFrame(summary).T
print("\n" + "="*65)
print(f"{'Metric':<15} | {'Euclidean (Mean +/- SD)':<25} | {'Haversine (Mean +/- SD)':<25}")
print("-" * 65)
for m in metrics_list:
    row = summary_df.loc[m]
    print(f"{m:<15} | {row['Euclidean_Mean']:.4f} +/- {row['Euclidean_SD']:.4f} | {row['Haversine_Mean']:.4f} +/- {row['Haversine_SD']:.4f}")
print("="*65)

In [ ]:
if not os.path.exists(OUTPUT_TABLES_PATH):
    os.makedirs(OUTPUT_TABLES_PATH)

summary_df.to_csv(os.path.join(OUTPUT_TABLES_PATH, "haversine_vs_euclidean_comparison.csv"))
print(f"Summary table successfully saved to: {os.path.join(OUTPUT_TABLES_PATH, 'haversine_vs_euclidean_comparison.csv')}")

In [ ]:
if not os.path.exists(OUTPUT_FIGURES_PATH):
    os.makedirs(OUTPUT_FIGURES_PATH)
    
plot_data = []
for metric_name, metrics_dict in results.items():
    for metric_key, values in metrics_dict.items():
        for val in values:
            plot_data.append({
                'Distance Metric': 'Euclidean Topology' if metric_name == 'euclidean' else 'Haversine Topology',
                'Metric': metric_key.replace('_', ' '),
                'Score': val
            })
df_plot = pd.DataFrame(plot_data)

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 18,
    'axes.labelsize': 20,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16,
    'axes.linewidth': 1.5,
    'figure.dpi': 300,
    'savefig.bbox': 'tight'
})
sns.set_theme(style="whitegrid", font="serif")

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=df_plot,
    x='Metric',
    y='Score',
    hue='Distance Metric',
    palette=['#777777', '#4285F4'],
    capsize=.08,
    errorbar='sd',
    edgecolor='black',
    linewidth=1.2,
    ax=ax
)

ax.set_ylabel("Metric Score", fontweight='bold')
ax.set_xlabel("Evaluation Metrics", fontweight='bold')
ymin = max(0, df_plot['Score'].min() - 0.1)
ax.set_ylim(ymin, 1.0)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=2, frameon=True, fancybox=False, shadow=False)
ax.yaxis.grid(True, linestyle='--', alpha=0.7)

plt.savefig(os.path.join(OUTPUT_FIGURES_PATH, "haversine_vs_euclidean_comparison.png"))
plt.show()
print(f"Comparative plot successfully saved to: {os.path.join(OUTPUT_FIGURES_PATH, 'haversine_vs_euclidean_comparison.png')}")

## Empirical Findings and Theoretical Insights

Wildfire events and geographic spots occur on Earth's spherical shell, meaning that coordinates (latitude and longitude) naturally follow a non-flat spherical spatial layout:
- **Euclidean Distance**: Operates under flat cartesian projections. While standardized coordinate representations are excellent node features, straight-line distance calculations degrade when applied to raw coordinates spanning large geographical distances due to Earth's curvature.
- **Haversine Distance**: Calculates the true great-circle distance along the surface of a spherical object in physical space (meters/kilometers). This matches the actual topological wildfire neighborhood distance metric.

This comparative analysis highlights the downstream impact of structural fidelity in spatial graph topology. The resulting table and plot show how the true physical metric compares against its Euclidean counterpart in multirelational wildfire prediction tasks.